In [ ]:
from selenium import webdriver
import time
import requests #fetches data from webpage
from bs4 import BeautifulSoup #extracts data we need
import pandas as pd #to store data as csv or excel format
from datetime import date
import re

WRITE_FILE_PATH = r'c:\Interest_rate_files'
CHROME_DRIVER_PATH = 'c:\\Interest_rate_files\\chromedriver-win64\\chromedriver.exe'
URL = "https://www.suryodaybank.com/rate-of-interest/#fixed-deposits" 

date1 = (date.today()).strftime('%d%m%Y') #Get today's date and convert it to to dd/mm/yy format
file_name= WRITE_FILE_PATH + '\Interest_rate_' + date1 + '.csv' #Set directory path to write file

# Set up the Chrome driver
driver = webdriver.Chrome(executable_path=CHROME_DRIVER_PATH)
driver.get(URL) # Fetch the HTML page
time.sleep(5)  ## Wait for 5 secs the JavaScript to execute (you can adjust the sleep time if needed)
html = driver.page_source # Get the rendered HTML
driver.quit() # Close the driver

soup=BeautifulSoup(html,"html.parser")
#print(soup.prettify())
table=soup.find("table")
rows=table.find_all("tr")

print("Today's date =",date1)
print("Interest rate file name = ",file_name)

list_of_interest_rates = []
tax_saver_row = []
row_count = 0
for i in rows[1:]:  #Skip heading
    data=i.find_all("td")
    row=['Suryoday Bank']
    row.extend([tr.text for tr in data]) # Extract the text from all the columns
    for i in range(0,5):
            #row[i] = ' '.join(row[i].strip().replace({'\n',@}, '').replace('\xa0', '').split())
            row[i] = ' '.join(re.sub(r'[\n@$#*\xa0]', '', row[i].strip()).split())
    if row_count == 12:
        tax_saver_row = ['Suryoday Bank', 'Tax-saver Fixed Deposit', row[2], row[3], row[4], row[5]]
        list_of_interest_rates.append(tax_saver_row)
    # new_row_items = ['','',30000000]
    #row.extend(new_row_items)
    list_of_interest_rates.append(row)
    row_count += 1

pd.set_option('display.max_columns', None)
int_rate_df=pd.DataFrame(list_of_interest_rates,columns=['bank_name','tenure','gen_rate','annualised_gen_rate',
                        'sr_rate','annualised_sr_rate'])
#print(list_of_interest_rates)
print(int_rate_df)
int_rate_df.to_csv(file_name, mode='a', header=False, index=False)
print("\nSuryoday Bank = Success. Number of rows added = ", len(list_of_interest_rates))